In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

# Generate car details
year = np.random.randint(2005, 2025, n)
km_driven = np.random.randint(5000, 200000, n)

fuel = np.random.choice(
    ["Petrol", "Diesel", "CNG"],
    n,
    p=[0.55, 0.35, 0.10]
)

seller_type = np.random.choice(
    ["Individual", "Dealer"],
    n
)

transmission = np.random.choice(
    ["Manual", "Automatic"],
    n,
    p=[0.7, 0.3]
)

owner = np.random.choice(
    ["First Owner", "Second Owner", "Third Owner"],
    n,
    p=[0.65, 0.25, 0.10]
)

present_price = np.random.uniform(3, 25, n)

# Calculate vehicle age
age = 2025 - year

# Generate selling prices (in lakhs)
selling_price = (
    present_price * (0.88 ** age)
    - km_driven * 0.000008
    + np.where(fuel == "Diesel", 0.5, 0)
    + np.where(transmission == "Automatic", 0.7, 0)
    + np.where(owner == "First Owner", 0.3, -0.3)
    + np.random.normal(0, 0.7, n)
)

# Ensure positive selling prices
selling_price = np.maximum(selling_price, 0.5)

# Add a few expensive-car outliers for Q10
outlier_indices = np.random.choice(n, 15, replace=False)
selling_price[outlier_indices] *= 2.5

# Create DataFrame
df = pd.DataFrame({
    "Year": year,
    "Present_Price": present_price.round(2),
    "Kms_Driven": km_driven,
    "Fuel_Type": fuel,
    "Seller_Type": seller_type,
    "Transmission": transmission,
    "Owner": owner,
    "Selling_Price": selling_price.round(2)
})

# Save dataset
df.to_csv("used_cars.csv", index=False)

print("Dataset created successfully!")
print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset created successfully!
   Year  Present_Price  Kms_Driven Fuel_Type Seller_Type Transmission  \
0  2011          10.83       13070    Petrol  Individual       Manual   
1  2024          17.97      131161    Petrol  Individual       Manual   
2  2019           9.03       13666    Petrol  Individual       Manual   
3  2015          21.21       62761    Petrol  Individual       Manual   
4  2012          19.28       41368    Diesel  Individual       Manual   

          Owner  Selling_Price  
0   First Owner           2.34  
1   First Owner          16.09  
2   First Owner           4.59  
3  Second Owner           4.47  
4   First Owner           4.57  

Shape: (1000, 8)

Columns: ['Year', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'Selling_Price']


In [ ]:
df = pd.read_csv("used_cars.csv").drop_duplicates().dropna()
df.head()

,Year,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Selling_Price
0,2011,10.83,13070,Petrol,Individual,Manual,First Owner,2.34
1,2024,17.97,131161,Petrol,Individual,Manual,First Owner,16.09
2,2019,9.03,13666,Petrol,Individual,Manual,First Owner,4.59
3,2015,21.21,62761,Petrol,Individual,Manual,Second Owner,4.47
4,2012,19.28,41368,Diesel,Individual,Manual,First Owner,4.57


In [ ]:
# baseline model without encoding only with numerical attributes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

x = df[["Year","Kms_Driven"]]
y = df["Selling_Price"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)
model = LinearRegression().fit(x_train, y_train)
pred = model.predict(x_test)

In [ ]:
# numerical features + one-hot encoding
df = df[["Year","Kms_Driven","Selling_Price","Transmission","Owner"]]
df_encoded = pd.get_dummies(df, columns=["Transmission", "Owner"], drop_first=True, dtype=int)
x_enc = df_encoded.drop(columns=["Selling_Price"])
y_enc = df_encoded["Selling_Price"]

x_enc_train, x_enc_test, y_enc_train, y_enc_test = train_test_split(x_enc,y_enc,test_size=0.2,random_state=42)
model_encoded = LinearRegression().fit(x_enc_train, y_enc_train)
pred_encoded = model_encoded.predict(x_enc_test)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

print(f"Numerical Only\tRMSE: {np.sqrt(mean_squared_error(y_test, pred))}\tR2: {r2_score(y_test, pred)}")
print(f"Encoded\t\tRMSE: {np.sqrt(mean_squared_error(y_enc_test, pred_encoded))}\tR2: {r2_score(y_enc_test, pred_encoded)}")

Numerical Only	RMSE: 2.7084903258188384	R2: 0.555676630719921
Encoded		RMSE: 2.713821681185308	R2: 0.5539257090655252
